In [1]:
import sys
import os

# Ajouter le répertoire database_manager au PYTHONPATH
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'database')))

# Importer le module db_operations
from DatabaseManager import DatabaseManager
from Doc2VecTrainer import Doc2VecTrainer
from pymongo import MongoClient

In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "../../artifacts/mlruns"

In [3]:
doc2vec_trainer = Doc2VecTrainer('staging')

In [4]:
client = MongoClient('localhost', 27017)
database_manager = DatabaseManager(client['github'])

In [5]:
train_df, test_df = database_manager.get_train_test_split_features()

In [7]:
t_df = train_df.iloc[:40]
te_df = test_df.iloc[:40]
t_df

,id,readme_preproc,others_preproc
0,37627792,"[img, align, right, width, 200, src, http, sta...","[document, head, manager, for, react, javascri..."
1,37659549,"[awesome, honeypots, awesome, honeypots, https...","[awesome, list, honeypot, resources, python, a..."
2,37680686,"[awesome, data, engineering, awesome, https, a...","[curated, list, data, engineering, tools, for,..."
3,37688007,"[swubanner, https, raw, githubusercontent, com...","[gem, collection, awesome, crystal, libraries,..."
4,37652385,"[cargo, edit, this, tool, extends, cargo, http...","[utility, for, managing, cargo, dependencies, ..."
5,37667292,[],"[dead, simple, opengl, cmake, cplusplus, graph..."
6,37647794,"[php, ipam, website, https, phpipam, net, desc...","[phpipam, development, repository, php, addres..."
7,37637797,"[elasticlunr, build, status, https, travis, or...","[based, lunr, but, more, flexible, and, custom..."
8,37637200,[],"[maps, laravel, eloquent, models, elasticsearc..."
9,37628106,"[pdcurses, stable, current, see, git, reposito...","[curses, library, for, environments, that, don..."


In [8]:
res = doc2vec_trainer.train(t_df, te_df)

2025/03/10 08:31:57 INFO mlflow.models.signature: Inferring model signature from type hints
/home/choux/miniconda3/envs/aag-talp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Registered model 'doc2vec_readme' already exists. Creating a new version of this model...
Created version '3' of model 'doc2vec_readme'.
2025/03/10 08:32:00 INFO mlflow.models.signature: Inferring model signature from type hints
Registered model 'doc2vec_others' already exists. Creating a new version of this model...
Created version '3' of model 'doc2vec_others'.
/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/src/model/Doc2VecTrainer.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentat

In [19]:
train_vectors = res[0]
test_vectors = res[1]
print(len(train_vectors))
print(len(test_vectors))
print(type(train_vectors))

4000
972
<class 'pandas.core.frame.DataFrame'>


In [21]:
database_manager.insert_repos_vectors_for_one_stage(train_vectors, 'staging')
database_manager.insert_repos_vectors_for_one_stage(test_vectors, 'staging')